In [2]:

import pandas as pd


In [3]:

data_path = "../data/raw/"

df = pd.read_csv(data_path + "language_learning_fluency.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (45000, 18)
Columns: ['learner_id', 'fsi_category', 'related_language', 'prior_languages', 'age_started', 'total_study_hours', 'active_use_share', 'immersion_months', 'speaking_practice', 'feedback_quality', 'comprehensible_input_hours', 'uses_srs', 'motivation', 'consistency', 'accent_nativelike', 'cefr_level', 'reached_fluency', 'dropped_out']


In [4]:

# Target columns for each model
targets = {
    "fluency": "reached_fluency",
    "dropout": "dropped_out",
    "cefr": "cefr_level",
    "study_hours": "total_study_hours",
    "accent": "accent_nativelike"
}

# Columns to exclude from each model

# Columns to exclude from each model
excluded_columns = {
    "fluency": [
        "learner_id",
        "reached_fluency",
        "cefr_level",
        "dropped_out"
    ],
    "dropout": [
        "learner_id",
        "dropped_out",
        "cefr_level",
        "reached_fluency"
    ],
    "cefr": [
        "learner_id",
        "cefr_level",
        "reached_fluency",
        "dropped_out"
    ],
    "study_hours": [
        "learner_id",
        "total_study_hours",
        "comprehensible_input_hours",
        "cefr_level",
        "reached_fluency",
        "dropped_out"
    ],
    "accent": [
        "learner_id",
        "accent_nativelike",
        "cefr_level",
        "reached_fluency",
        "dropped_out"
    ]
}

print("Targets and exclusions updated successfully.")



Targets and exclusions updated successfully.


In [5]:

feature_sets = {}

for model_name, target_column in targets.items():

    feature_sets[model_name] = [
        column for column in df.columns
        if column not in excluded_columns[model_name]
    ]

    print(f"\n{model_name.upper()} MODEL")
    print("Target:", target_column)
    print("Number of features:", len(feature_sets[model_name]))
    print("Features:", feature_sets[model_name])



FLUENCY MODEL
Target: reached_fluency
Number of features: 14
Features: ['fsi_category', 'related_language', 'prior_languages', 'age_started', 'total_study_hours', 'active_use_share', 'immersion_months', 'speaking_practice', 'feedback_quality', 'comprehensible_input_hours', 'uses_srs', 'motivation', 'consistency', 'accent_nativelike']

DROPOUT MODEL
Target: dropped_out
Number of features: 14
Features: ['fsi_category', 'related_language', 'prior_languages', 'age_started', 'total_study_hours', 'active_use_share', 'immersion_months', 'speaking_practice', 'feedback_quality', 'comprehensible_input_hours', 'uses_srs', 'motivation', 'consistency', 'accent_nativelike']

CEFR MODEL
Target: cefr_level
Number of features: 14
Features: ['fsi_category', 'related_language', 'prior_languages', 'age_started', 'total_study_hours', 'active_use_share', 'immersion_months', 'speaking_practice', 'feedback_quality', 'comprehensible_input_hours', 'uses_srs', 'motivation', 'consistency', 'accent_nativelike']



In [6]:

for model_name, target_column in targets.items():

    features = feature_sets[model_name]

    print(f"\n{model_name.upper()} MODEL")
    print("Target:", target_column)

    # Check that the target is not in the input features
    print("Target excluded from features:", target_column not in features)

    # Check that learner_id is excluded
    print("Learner ID excluded:", "learner_id" not in features)

    # Check the number of features
    print("Number of features:", len(features))



FLUENCY MODEL
Target: reached_fluency
Target excluded from features: True
Learner ID excluded: True
Number of features: 14

DROPOUT MODEL
Target: dropped_out
Target excluded from features: True
Learner ID excluded: True
Number of features: 14

CEFR MODEL
Target: cefr_level
Target excluded from features: True
Learner ID excluded: True
Number of features: 14

STUDY_HOURS MODEL
Target: total_study_hours
Target excluded from features: True
Learner ID excluded: True
Number of features: 12

ACCENT MODEL
Target: accent_nativelike
Target excluded from features: True
Learner ID excluded: True
Number of features: 13


In [7]:

# Check whether any target columns appear in the feature sets

for model_name, features in feature_sets.items():
    target_columns = list(targets.values())

    remaining_targets = [
        column for column in target_columns
        if column in features
    ]

    print(f"\n{model_name.upper()} MODEL")
    print("Other target columns present:", remaining_targets)



FLUENCY MODEL
Other target columns present: ['total_study_hours', 'accent_nativelike']

DROPOUT MODEL
Other target columns present: ['total_study_hours', 'accent_nativelike']

CEFR MODEL
Other target columns present: ['total_study_hours', 'accent_nativelike']

STUDY_HOURS MODEL
Other target columns present: ['accent_nativelike']

ACCENT MODEL
Other target columns present: ['total_study_hours']


In [8]:

# Separate input features (X) and target values (y) for each model

X_fluency = df[feature_sets["fluency"]]
y_fluency = df[targets["fluency"]]

X_dropout = df[feature_sets["dropout"]]
y_dropout = df[targets["dropout"]]

X_cefr = df[feature_sets["cefr"]]
y_cefr = df[targets["cefr"]]

X_study_hours = df[feature_sets["study_hours"]]
y_study_hours = df[targets["study_hours"]]

X_accent = df[feature_sets["accent"]]
y_accent = df[targets["accent"]]

print("Input and target variables created successfully.")


Input and target variables created successfully.


In [9]:

# Check the number of rows and columns for each model

print("Fluency:", X_fluency.shape, y_fluency.shape)
print("Dropout:", X_dropout.shape, y_dropout.shape)
print("CEFR:", X_cefr.shape, y_cefr.shape)
print("Study hours:", X_study_hours.shape, y_study_hours.shape)
print("Accent:", X_accent.shape, y_accent.shape)


Fluency: (45000, 14) (45000,)
Dropout: (45000, 14) (45000,)
CEFR: (45000, 14) (45000,)
Study hours: (45000, 12) (45000,)
Accent: (45000, 13) (45000,)


In [10]:

from sklearn.model_selection import train_test_split


In [11]:

X_train_fluency, X_test_fluency, y_train_fluency, y_test_fluency = train_test_split(
    X_fluency,
    y_fluency,
    test_size=0.2,
    random_state=42,
    stratify=y_fluency
)

print("Training features:", X_train_fluency.shape)
print("Testing features:", X_test_fluency.shape)
print("Training target:", y_train_fluency.shape)
print("Testing target:", y_test_fluency.shape)


Training features: (36000, 14)
Testing features: (9000, 14)
Training target: (36000,)
Testing target: (9000,)


In [12]:

print("Original target proportions:")
print(y_fluency.value_counts(normalize=True).sort_index())

print("\nTraining target proportions:")
print(y_train_fluency.value_counts(normalize=True).sort_index())

print("\nTesting target proportions:")
print(y_test_fluency.value_counts(normalize=True).sort_index())


Original target proportions:
reached_fluency
0    0.642378
1    0.357622
Name: proportion, dtype: float64

Training target proportions:
reached_fluency
0    0.642389
1    0.357611
Name: proportion, dtype: float64

Testing target proportions:
reached_fluency
0    0.642333
1    0.357667
Name: proportion, dtype: float64


In [13]:

# Dropout classification
X_train_dropout, X_test_dropout, y_train_dropout, y_test_dropout = train_test_split(
    X_dropout, y_dropout,
    test_size=0.2,
    random_state=42,
    stratify=y_dropout
)

# CEFR classification
X_train_cefr, X_test_cefr, y_train_cefr, y_test_cefr = train_test_split(
    X_cefr, y_cefr,
    test_size=0.2,
    random_state=42,
    stratify=y_cefr
)

# Study-hours regression
X_train_study_hours, X_test_study_hours, y_train_study_hours, y_test_study_hours = train_test_split(
    X_study_hours, y_study_hours,
    test_size=0.2,
    random_state=42
)

# Accent regression
X_train_accent, X_test_accent, y_train_accent, y_test_accent = train_test_split(
    X_accent, y_accent,
    test_size=0.2,
    random_state=42
)

print("All five datasets have been split successfully.")


All five datasets have been split successfully.


In [14]:

print("Dropout:", X_train_dropout.shape, X_test_dropout.shape)
print("CEFR:", X_train_cefr.shape, X_test_cefr.shape)
print("Study hours:", X_train_study_hours.shape, X_test_study_hours.shape)
print("Accent:", X_train_accent.shape, X_test_accent.shape)


Dropout: (36000, 14) (9000, 14)
CEFR: (36000, 14) (9000, 14)
Study hours: (36000, 12) (9000, 12)
Accent: (36000, 13) (9000, 13)
